# OCC 历史 SOXX 认购／认沽量的两次精确公开来源探针

只读取2019-05-31一个历史日的固定C/P两个OCC批处理URL；保存原始响应字节、哈希与NAS回读。不推断多年覆盖、发布时间或因子有效性。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import csv,gzip,hashlib,io,json,tempfile,urllib.error,urllib.request
from datetime import datetime,timezone
from pathlib import Path
import qlib
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence

root=Path.cwd();card=root/"research/occ_soxx_put_call_source_probe_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());assert plan["frozen"]
sha=lambda b:hashlib.sha256(b).hexdigest()
snapshot=root/plan["source_storage"]["local_exact_bytes"];snapshot.mkdir(parents=True,exist_ok=True)
records=[];bodies={}
for kind in ("call","put"):
 url=plan["exact_queries"][kind]
 started=datetime.now(timezone.utc).isoformat()
 request=urllib.request.Request(url,headers={"User-Agent":plan["transport"]["user_agent"],"Accept":"text/csv,*/*"},method="GET")
 status=None;headers={};final_url=url;error=None;body=b"";oversize=False
 try:
  with urllib.request.urlopen(request,timeout=plan["transport"]["timeout_seconds_each"]) as response:
   status=response.status;headers=dict(response.headers.items());final_url=response.geturl()
   body=response.read(plan["transport"]["max_response_bytes_each"]+1)
 except urllib.error.HTTPError as exc:
  status=exc.code;headers=dict(exc.headers.items());final_url=exc.geturl()
  body=exc.read(plan["transport"]["max_response_bytes_each"]+1)
 except Exception as exc:
  error=f"{type(exc).__name__}: {str(exc)[:250]}"
 received=datetime.now(timezone.utc).isoformat()
 if len(body)>plan["transport"]["max_response_bytes_each"]:
  oversize=True;body=body[:plan["transport"]["max_response_bytes_each"]]
 raw_path=snapshot/(kind+".response")
 raw_path.write_bytes(body)
 bodies[kind]=body
 selected={k:next((v for key,v in headers.items() if key.lower()==k.lower()),None) for k in ["Date","Last-Modified","Content-Type","Content-Disposition","Cache-Control","Content-Encoding"]}
 parsed_rows=0;positive_qty=0;record_layout_rows=0;decode_error=None
 if body and not oversize:
  try:
   data=gzip.decompress(body) if body.startswith(bytes.fromhex("1f8b")) else body
   lines=csv.reader(io.StringIO(data.decode("utf-8-sig")))
   for row in lines:
    if len(row)<8:continue
    try:qty=int(row[0].replace(",","").strip())
    except ValueError:continue
    record_layout_rows+=1
    underlying=row[1].strip().upper();side=row[4].strip().upper();activity=row[6].strip()
    if underlying=="SOXX" and side==("C" if kind=="call" else "P") and activity in {"05/31/2019","2019-05-31","20190531"}:
     parsed_rows+=1;positive_qty+=qty
  except Exception as exc:
   decode_error=f"{type(exc).__name__}: {str(exc)[:250]}"
 records.append({"kind":kind,"requested_url":url,"final_url":final_url,"requested_at_utc":started,"received_at_utc":received,
                 "http_status":status,"selected_headers":selected,"response_path":str(raw_path.relative_to(root)),
                 "response_bytes":len(body),"response_sha256":sha(body),"response_is_complete":not oversize and error is None,
                 "transport_error":error,"decode_error":decode_error,"record_layout_rows":record_layout_rows,
                 "SOXX_expected_side_date_rows":parsed_rows,"SOXX_expected_side_contracts":positive_qty})
receipt={"plan_sha256":sha(plan_path.read_bytes()),"HTTP_GET_attempts":2,"records":records,
         "historical_first_public_qualified":False,"historical_revision_qualified":False,
         "SOXX_outcome_reads":0,"model_fits":0,"native_accounts":0}
(card/"source_receipt.json").write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+"\n")
qlib.init(provider_uri=str((root/"data/qlib/minute_information_pit_allocator_20261012/daily_prices").resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.start(experiment_name=card.name,uri=str((card/"qlib_recorder").resolve())):
 rec=R.get_recorder()
 rec.save_objects(artifact_path="sources",call_body=bodies["call"],put_body=bodies["put"],receipt=receipt)
 R.log_params(source="OCC Volume Query",report_date=plan["target_report_date"],underlying="SOXX")
 R.log_metrics(source_GETs=2,valid_SOXX_rows=sum(x["SOXX_expected_side_date_rows"] for x in records),
               SOXX_contracts=sum(x["SOXX_expected_side_contracts"] for x in records))
 rid,eid=rec.id,str(rec.experiment_id)
parent_binding=json.loads((root/"research/minute_information_pit_allocator_20261012/nas_binding.json").read_text())
binding={"schema_version":"occ_source_recorder_backup_v1","authority":"yanxuan_nas","ssh":parent_binding["ssh"],"source_id":parent_binding["source_id"]}
binding_path=card/"nas_binding.json";binding_path.write_text(json.dumps(binding,ensure_ascii=False,indent=2)+"\n")
nas_evidence=None;nas_error=None;readback_verified=False
try:
 nas_evidence=archive_nas_recorder(binding_path,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
 with tempfile.TemporaryDirectory(prefix="occ-soxx-readback-") as temp:
  dest=Path(temp)
  materialize_nas_evidence(binding_path,card=dest)
  with R.uri_context(str((dest/"qlib_recorder").resolve())):
   old=R.get_recorder(experiment_id=eid,recorder_id=rid)
   restored={"call":old.load_object("sources/call_body"),"put":old.load_object("sources/put_body")}
  readback_verified=all(restored[kind]==bodies[kind] for kind in ("call","put"))
except Exception as exc:
 nas_error=f"{type(exc).__name__}: {str(exc)[:300]}"
source_gate=all(x["http_status"]==200 and x["response_is_complete"] and x["SOXX_expected_side_date_rows"]>0 and x["SOXX_expected_side_contracts"]>0 for x in records)
status=("one_2019_SOXX_public_call_put_source_observed" if source_gate and readback_verified else
        "needs_NAS_backup_or_readback" if source_gate else "fixed_old_date_OCC_source_query_failed")
result={"status":status,"decision":status,"plan_sha256":sha(plan_path.read_bytes()),"source_receipt":"source_receipt.json",
        "source_gate_pass":source_gate,"NAS_readback_verified":readback_verified,"NAS_error":nas_error,"NAS_evidence":nas_evidence,
        "recorder_id":rid,"experiment_id":eid,"response_sha256":{x["kind"]:x["response_sha256"] for x in records},
        "HTTP_GETs":2,"SOXX_outcome_reads":0,"model_fits":0,"native_accounts":0,"historical_first_public_qualified":False,
        "historical_2011_2022_coverage_qualified":False,"option_IV_skew_qualified":False,"independent_forward_days":0,
        "next_action":"If both old-date sides are real and NAS verified, freeze a separate sparse-date coverage/revision pilot before any factor. Otherwise close these exact URLs and preserve the concrete response/transport dependency."}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"status":status,"records":[{k:x[k] for k in ("kind","http_status","response_bytes","record_layout_rows","SOXX_expected_side_date_rows","SOXX_expected_side_contracts") } for x in records],"NAS_readback_verified":readback_verified,"NAS_error":nas_error})


{'status': 'fixed_old_date_OCC_source_query_failed', 'records': [{'kind': 'call', 'http_status': 200, 'response_bytes': 41, 'record_layout_rows': 0, 'SOXX_expected_side_date_rows': 0, 'SOXX_expected_side_contracts': 0}, {'kind': 'put', 'http_status': 200, 'response_bytes': 41, 'record_layout_rows': 0, 'SOXX_expected_side_date_rows': 0, 'SOXX_expected_side_contracts': 0}], 'NAS_readback_verified': True, 'NAS_error': None}


## 中文结论

OCC官方批处理说明允许按期权标的、日频、认购／认沽字段查询；其网页同时提示成交量查询仅提供过去24个月。按冻结计划只对 **2019-05-31、SOXX标的**各发一次认购和认沽只读GET，没有改参数或重试。两个请求均返回HTTP200，但正文都是相同的41字节：`Report date cannot be prior to  2 years`，不是CSV，也没有合格SOXX合约行。精确URL、UTC请求／响应时间、头、原始字节和SHA-256见 `source_receipt.json` 及 `data/snapshots/occ_soxx_put_call_source_probe_20261024/`。

来源门失败，决定 `fixed_old_date_OCC_source_query_failed_two_year_public_age_gate`。这个公开OCC Volume Query批处理地址不能按本次精确查询提供2019历史日，更不能证明2011–2022 SOXX认购／认沽序列、历史首次送达或修订。它不否定其他尚未测试的合法公开或授权归档，也不是对SOXX期权25-Delta隐含波动率的检验。用户当前不联系供应商；本卡没有询价、购买、订阅或读取账户。

两份原始响应均逐字节保留，Qlib Recorder归档NAS并对原始body做限定恢复回读，远端SHA已核对；大型Recorder仅在本地及NAS保留，未将自动生成的无关工作区代码快照提交到Git。0市场结果、0拟合、0资金账户、0独立前向日。该固定URL版本关闭；SOXX目标仍未达到。
